In [ ]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — data transformation

**Step 1: load & display.** Reads the *latest* versioned CSV per table from
`query_result_data/` and displays it. The raw extracts are treated as
**read-only** — nothing here writes back to that folder.

All loading logic lives in `transform.py` (libraries it imports: `pandas`,
stdlib `re`/`pathlib`, and `data_validation` for the shared query-id → label map).
Edit `TABLE` to choose which table to display; `PREVIEW_ROWS` sets the preview size.

In [43]:
# Cell 1 — imports + notebook-level settings
import pandas as pd
from IPython.display import display
import transform as tf
import data_validation as dv
import adv_validation as adv

PREVIEW_ROWS = 10  # rows shown in the inline preview (notebook-only setting)
# Cell 2 — see the available tables (label <- latest file)
for label, filename in sorted(tf.list_tables().items()):
    print(f"{label:30s} <- {filename}")

borrow_repay                   <- borrow_repay_7798273_20260624T081133Z.csv
collateral_toggle              <- collateral_toggle_7798372_20260624T082649Z.csv
flashloan                      <- flashloan_7798349_20260624T082343Z.csv
liquidation                    <- liquidation_7798339_20260624T082345Z.csv
oracle_price_usd_eth_weth_6h   <- oracle_price_usd_eth_weth_6h_7798226_20260624T092854Z.csv
query_7711171                  <- decimal_reference_part1_7711171_20260623T190322Z.csv
query_7711265                  <- decimal_reference_part2_7711265_20260623T191755Z.csv
query_7711276                  <- decimal_reference_part3_7711276_20260623T192309Z.csv
query_7711290                  <- decimal_reference_part4_7711290_20260623T192311Z.csv
query_7711298                  <- decimal_reference_part5_7711298_20260623T192320Z.csv
query_7711304                  <- decimal_reference_part6_7711304_20260623T192322Z.csv
query_7711307                  <- decimal_reference_part9_collateral_toggle_77113

In [44]:
# Cell 3 — load one table (read-only) and display it
TABLE = "supply_withdraw"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_supply_withdraw = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_supply_withdraw.shape[0]} rows x {df_0_supply_withdraw.shape[1]} cols")
display(df_0_supply_withdraw.dtypes.to_frame("dtype"))
display(df_0_supply_withdraw.head(PREVIEW_ROWS))

supply_withdraw: 84630 rows x 12 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
supply_amount_raw,object
withdrawal_amount_raw,object
net_supply_flow_raw,object
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64


,time_bucket,asset,asset_symbol,supply_amount_raw,withdrawal_amount_raw,net_supply_flow_raw,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,latest_supply_block,latest_withdraw_block
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,80937905573546271332,3352481644830690207,77585423928715581125,4,1,3,1,22170797.0,22170797.0
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,720471761,246464895,474006866,12,2,5,1,22170878.0,22170544.0
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,20000000000000000000,255756260689565740664,-235756260689565740664,1,1,1,1,22170813.0,22170460.0
3,2025-04-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,101772080173949086951,NaN,101772080173949086951,2,0,2,0,22170680.0,NaN
4,2025-04-01 00:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,18242549021403584259,100920688337165907518,-82678139315762323259,1,1,1,1,22170885.0,22170883.0
5,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,NaN,390160229791931868145049,-390160229791931868145049,0,1,0,1,NaN,22170539.0
6,2025-04-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,584566561797,543123699307,41442862490,1,5,1,1,22170836.0,22170922.0
7,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,22598482970008222347,26951146895841088925,-4352663925832866578,3,1,2,1,22170833.0,22170805.0
8,2025-04-01 00:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,83734342036557266,1923275558078465767,-1839541216041908501,1,3,1,3,22170502.0,22170696.0
9,2025-04-01 00:00:00.000 UTC,0x9d39a5de30e57443bff2a8307a4256c8797a3497,sUSDe,NaN,2022925936809196800000000,-2022925936809196800000000,0,1,0,1,NaN,22170862.0


In [45]:
#7711171
# df_1_supply_withdraw = tf.load_table("query_7711171")
# print(f"{TABLE}: {df_1_supply_withdraw.shape[0]} rows x {df_1_supply_withdraw.shape[1]} cols")
# display(df_1_supply_withdraw.dtypes.to_frame("dtype"))
# display(df_1_supply_withdraw.head(PREVIEW_ROWS))
df_0_supply_withdraw.drop(["net_supply_flow_raw", "latest_supply_block", "latest_withdraw_block"], axis=1, inplace=True)
# df_1_supply_withdraw.drop(["metric", "unit"], axis=1, inplace=True)
# df_1_supply_withdraw = df_1_supply_withdraw.drop(df_1_supply_withdraw.index[[0, 1]])

#query_7798226 =  7714873
df_oracle_price_usd_eth = tf.load_table("oracle_price_usd_eth_weth_6h")
print(f"{TABLE}: {df_oracle_price_usd_eth.shape[0]} rows x {df_oracle_price_usd_eth.shape[1]} cols")
display(df_oracle_price_usd_eth.dtypes.to_frame("dtype"))
df_oracle_price_usd_eth


supply_withdraw: 205492 rows x 7 cols


,dtype
time_bucket,str
asset,str
symbol,str
decimals,int64
avg_price_usd,float64
avg_price_eth,float64
price_points,int64


,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
205487,2026-03-30 22:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
205488,2026-03-30 22:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
205489,2026-03-30 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000145,0.000493,2
205490,2026-03-30 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [46]:

stats = adv.statistical_validation(df_oracle_price_usd_eth, columns=["avg_price_usd", "avg_price_eth"],save=False)
print(stats[["column", "n_null", "null_pct", "n_zero", "zero_pct"]])

          column  n_null  null_pct  n_zero  zero_pct
0  avg_price_usd       0       0.0       0       0.0
1  avg_price_eth       0       0.0       0       0.0


In [47]:
# Cell — divide raw amount columns by 10**decimals -> real token units
# decimals come from the reference table (df_1); raw_token_amount rows only.
df_0_supply_withdraw_scaled = tf.scale_by_decimals(df_0_supply_withdraw, df_oracle_price_usd_eth, drop_raw=True)

print(f"raw cols scaled: {tf.raw_amount_columns(df_0_supply_withdraw)}")
display(df_0_supply_withdraw_scaled.head(PREVIEW_ROWS))
# Cell — validation checks on the scaled amount columns of df_0 (after scale_by_decimals)
import adv_validation as av

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_supply_withdraw)]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_supply_withdraw_scaled, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_0_supply_withdraw_scaled, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_0_supply_withdraw_scaled, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")

raw cols scaled: ['supply_amount_raw', 'withdrawal_amount_raw']


,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,4,1,3,1,80.937906,3.352482e+00
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,12,2,5,1,7.204718,2.464649e+00
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,1,1,1,1,20.000000,2.557563e+02
3,2025-04-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,2,0,2,0,101.772080,NaN
4,2025-04-01 00:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,1,1,1,1,18.242549,1.009207e+02
5,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0,1,0,1,NaN,3.901602e+05
6,2025-04-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,1,5,1,1,584566.561797,5.431237e+05
7,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,3,1,2,1,22.598483,2.695115e+01
8,2025-04-01 00:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,1,3,1,3,0.083734,1.923276e+00
9,2025-04-01 00:00:00.000 UTC,0x9d39a5de30e57443bff2a8307a4256c8797a3497,sUSDe,0,1,0,1,NaN,2.022926e+06


amount columns: ['supply_amount', 'withdrawal_amount']

===== supply_amount =====
negatives : 0 of 66842 (0.0%)

===== withdrawal_amount =====
negatives : 0 of 62536 (0.0%)


In [48]:
amount_cols = [
    "supply_amount",
    "withdrawal_amount",
]
df_3_supply_withdraw = tf.multiply_by_price(df_0_supply_withdraw_scaled, df_oracle_price_usd_eth, amount_cols)
df_3_supply_withdraw

,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,4,1,3,1,80.937906,3.352482e+00,485.290192,0.265625,20.100921,0.011002
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,12,2,5,1,7.204718,2.464649e+00,594779.182181,325.553748,203466.945715,111.368099
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,1,1,1,1,20.000000,2.557563e+02,20.012268,0.010954,255.913146,0.140075
3,2025-04-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,2,0,2,0,101.772080,NaN,1381.047128,0.755919,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,1,1,1,1,18.242549,1.009207e+02,18.275582,0.010003,101.103430,0.055339
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84625,2026-03-31 22:00:00.000 UTC,0xae78736cd615f374d3085123a210448e74fc6393,rETH,1,0,1,0,0.629005,NaN,NaN,NaN,NaN,NaN
84626,2026-03-31 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,40,25,14,10,2917.037666,2.196790e+04,NaN,NaN,NaN,NaN
84627,2026-03-31 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,4,4,3,4,0.007432,1.199500e+00,NaN,NaN,NaN,NaN
84628,2026-03-31 22:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,9,25,6,19,877742.728767,2.560279e+06,NaN,NaN,NaN,NaN


In [49]:
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]

df_4_supply_withdraw = tf.aggregate_by_time_bucket(df_3_supply_withdraw, "time_bucket", amount_cols, agg_func='sum')
df_4_supply_withdraw

,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-04-01 00:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287
1,2025-04-01 02:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741
2,2025-04-01 04:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778
3,2025-04-01 06:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607
4,2025-04-01 08:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327
...,...,...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,228,157,121,104,0.000000e+00,0.000000,0.000000e+00,0.000000
4376,2026-03-31 16:00:00.000 UTC,223,193,94,123,0.000000e+00,0.000000,0.000000e+00,0.000000
4377,2026-03-31 18:00:00.000 UTC,164,155,102,99,0.000000e+00,0.000000,0.000000e+00,0.000000
4378,2026-03-31 20:00:00.000 UTC,168,159,113,103,0.000000e+00,0.000000,0.000000e+00,0.000000


In [50]:

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")


    neg = adv.negative_value_check(df_4_supply_withdraw, col, plot=False)
    # rng = av.range_check(df_4_supply_withdraw, col, plot=False)
    # dev = av.deviation_score(df_4_supply_withdraw, col, plot=False)

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")


amount columns: ['supply_tx_count', 'withdrawal_tx_count', 'unique_suppliers', 'unique_withdraw_users', 'supply_amount_value_usd', 'supply_amount_value_eth', 'withdrawal_amount_value_usd', 'withdrawal_amount_value_eth']

===== supply_tx_count =====
negatives : 0 of 4380 (0.0%)

===== withdrawal_tx_count =====
negatives : 0 of 4380 (0.0%)

===== unique_suppliers =====
negatives : 0 of 4380 (0.0%)

===== unique_withdraw_users =====
negatives : 0 of 4380 (0.0%)

===== supply_amount_value_usd =====
negatives : 0 of 4380 (0.0%)

===== supply_amount_value_eth =====
negatives : 0 of 4380 (0.0%)

===== withdrawal_amount_value_usd =====
negatives : 0 of 4380 (0.0%)

===== withdrawal_amount_value_eth =====
negatives : 0 of 4380 (0.0%)


In [51]:
stats = adv.statistical_validation(df_4_supply_withdraw, amount_cols, save=False)

print(stats[["column", 
             "null_pct", "n_zero", "zero_pct", "negative_pct",
    "n_sentinel", "mean", "std", "cv",
    # "min", "p01", "p05", "q25", "median", "q75", 
    # "p95", "p99", #"max",
    # "iqr", "mad", "skewness", "excess_kurtosis",
    # "n_outliers_iqr", "outlier_iqr_pct", "n_outliers_mad", "outlier_mad_pct",
    "heavy_tailed", "success",]])

                        column  null_pct  n_zero  zero_pct  negative_pct  \
0              supply_tx_count       0.0       0     0.000           0.0   
1          withdrawal_tx_count       0.0       0     0.000           0.0   
2             unique_suppliers       0.0       0     0.000           0.0   
3        unique_withdraw_users       0.0       0     0.000           0.0   
4      supply_amount_value_usd       0.0      12     0.274           0.0   
5      supply_amount_value_eth       0.0      12     0.274           0.0   
6  withdrawal_amount_value_usd       0.0      12     0.274           0.0   
7  withdrawal_amount_value_eth       0.0      12     0.274           0.0   

   n_sentinel          mean           std        cv  heavy_tailed  success  
0           0  1.851420e+02  9.858905e+01  0.532505          True     True  
1           0  1.557708e+02  8.967866e+01  0.575709          True     True  
2           0  9.767237e+01  4.646847e+01  0.475759          True     True  
3      

In [52]:
# load one table (read-only) and display it
# THE VALUES ARE NOT RAW, IT WAS PRE SCALED BY SQL
TABLE = "borrow_repay"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_borrow = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_borrow.shape[0]} rows x {df_0_borrow.shape[1]} cols")
display(df_0_borrow.dtypes.to_frame("dtype"))
display(df_0_borrow.head(PREVIEW_ROWS))

borrow_repay: 54198 rows x 15 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
borrow_amount,float64
repay_amount,float64
net_debt_flow,float64
borrow_tx_count,int64
repay_tx_count,int64
stable_borrow_tx_count,int64
variable_borrow_tx_count,int64


,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,net_debt_flow,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043e+00,-6.600429e-01,2,4,0,2,1,3,5102958609642485824368628,22170766.0,22170811.0
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1.129596e+03,3.931117e+04,3,2,0,3,3,2,45000000000000000000000000,22170868.0,22170845.0
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,2.000000e+01,4.000000e+02,2,1,0,2,2,1,33221773480495132622398080,22170778.0,22170797.0
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1.500000e+05,1,0,0,1,1,0,48094819243714186856515190,22170769.0,NaN
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,2.000000e+02,1,0,0,1,1,0,4967538120472516497064990,22170544.0,NaN
5,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,8.277720e+06,1.948000e+07,-1.120228e+07,16,17,0,16,11,10,46030830777063318637077796,22170914.0,22170930.0
6,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,1.789983e+04,4.820925e+03,1.307890e+04,1,2,0,1,1,2,89416998540159967524840963,22170789.0,22170928.0
7,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,6.983152e+02,5.939031e+02,1.044121e+02,14,13,0,14,3,4,26734561820823454479814960,22170932.0,22170930.0
8,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,9.109636e+04,7.834299e+06,-7.743202e+06,14,7,0,14,14,4,48260450698177106607739218,22170925.0,22170917.0
9,2025-04-01 00:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,5.020625e+04,-5.020625e+04,0,1,0,0,0,1,NaN,NaN,22170524.0


In [53]:
# df_1 = tf.load_table("query_7711265")
# print(f"{TABLE}: {df_1.shape[0]} rows x {df_1.shape[1]} cols")
# display(df_1.dtypes.to_frame("dtype"))
# display(df_1.head(PREVIEW_ROWS))

df_0_borrow["last_borrow_rate"] = df_0_borrow["last_borrow_rate"].astype(float) / 10**27
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,net_debt_flow,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,-6.600429e-01,2,4,0,2,1,3,0.005103,22170766.0,22170811.0
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3.931117e+04,3,2,0,3,3,2,0.045000,22170868.0,22170845.0
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,4.000000e+02,2,1,0,2,2,1,0.033222,22170778.0,22170797.0
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1.500000e+05,1,0,0,1,1,0,0.048095,22170769.0,NaN
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,2.000000e+02,1,0,0,1,1,0,0.004968,22170544.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,3.641508e+06,24,18,0,24,22,15,0.034642,24773806.0,24773857.0
54194,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,-1.420239e+02,7,4,0,7,4,4,0.022331,24773853.0,24773817.0
54195,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,1.500000e+02,2,0,0,2,2,0,0.022250,24773674.0,NaN
54196,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,-1.000000e-08,1,1,0,1,1,1,0.003051,24773579.0,24773579.0


In [54]:
#oracle_price_usd_eth_weth_6h =  7714873
df_oracle_price_usd_eth 

,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
205487,2026-03-30 22:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
205488,2026-03-30 22:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
205489,2026-03-30 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000145,0.000493,2
205490,2026-03-30 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [55]:
df_0_borrow.drop(["net_debt_flow","latest_borrow_block", "latest_repay_block"], axis=1, inplace=True)
DF_common = df_0_borrow[["time_bucket", "asset", "asset_symbol", "last_borrow_rate"]].copy()
df_0_borrow.drop(["last_borrow_rate"], axis=1, inplace=True)
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,2,4,0,2,1,3
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3,2,0,3,3,2
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,2,1,0,2,2,1
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1,0,0,1,1,0
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,24,18,0,24,22,15
54194,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,7,4,0,7,4,4
54195,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,2,0,0,2,2,0
54196,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,1,1,0,1,1,1


In [56]:
# Cell — divide borrow_repay raw amount columns by 10**decimals -> real token units
# decimals come from df_1_borrow (the oracle_price table's `decimals` column), per asset.
# df_2_borrow = tf.scale_by_decimals(df_0_borrow, df_1_borrow)

# print(f"raw cols scaled: {tf.raw_amount_columns(df_0_borrow)}")
# display(df_2_borrow.head(PREVIEW_ROWS))

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
# amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_borrow)]
# print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)

amount_cols = [
    "borrow_amount",
    "repay_amount",
]
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_borrow, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_2_borrow, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_2_borrow, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")




===== borrow_amount =====
negatives : 0 of 45301 (0.0%)

===== repay_amount =====
negatives : 0 of 44818 (0.0%)


In [57]:


df_2_borrow = tf.multiply_by_price(df_0_borrow, df_oracle_price_usd_eth, amount_cols)
df_2_borrow



,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,2,4,0,2,1,3,3.302165e+04,18.074477,87510.915129,47.899300
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3,2,0,3,3,2,4.045823e+04,22.144902,1130.083857,0.618554
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,2,1,0,2,2,1,4.202576e+02,0.230029,20.012273,0.010954
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1,0,0,1,1,0,1.500965e+05,82.155660,NaN,NaN
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,1,0,0,1,1,0,4.376308e+05,239.538168,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,24,18,0,24,22,15,4.184672e+06,2062.457100,542958.580153,267.602510
54194,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,7,4,0,7,4,4,2.190643e+05,107.967016,507229.767282,249.990952
54195,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,2,0,0,2,2,0,1.499275e+02,0.073893,NaN,NaN
54196,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,1,1,0,1,1,1,3.543657e+02,0.174652,354.366381,0.174652


In [58]:
borrow_agg_cols = [
    "borrow_tx_count",
    "repay_tx_count",
    # "stable_borrow_tx_count",
    "variable_borrow_tx_count",
    "unique_borrowers",
    "unique_repayers",
    "borrow_amount_value_usd",
    "borrow_amount_value_eth",
    "repay_amount_value_usd",
    "repay_amount_value_eth",
]

df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')
# df_3_borrow.drop(["stable_borrow_tx_count"], axis=1, inplace=True )

df_3_borrow

,time_bucket,borrow_tx_count,repay_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-04-01 00:00:00.000 UTC,54,47,54,37,27,1.032549e+07,5651.683080,2.855941e+07,15632.055680
1,2025-04-01 02:00:00.000 UTC,58,58,58,47,42,1.360886e+07,7414.203082,2.294215e+07,12499.053578
2,2025-04-01 04:00:00.000 UTC,48,45,48,37,38,9.805061e+06,5326.065746,1.644968e+07,8935.388910
3,2025-04-01 06:00:00.000 UTC,64,67,64,43,36,2.930883e+07,15808.263478,5.574171e+07,30065.281708
4,2025-04-01 08:00:00.000 UTC,95,71,95,67,42,1.601311e+07,8520.523856,3.426404e+07,18231.785356
...,...,...,...,...,...,...,...,...,...,...
4363,2026-03-30 14:00:00.000 UTC,112,101,112,86,68,5.067923e+07,24561.015042,2.762900e+07,13390.077453
4364,2026-03-30 16:00:00.000 UTC,113,97,113,83,62,1.125271e+07,5467.748299,1.729953e+07,8405.886893
4365,2026-03-30 18:00:00.000 UTC,104,92,104,74,72,9.916947e+06,4888.783277,9.870147e+06,4865.700859
4366,2026-03-30 20:00:00.000 UTC,75,80,75,61,57,4.707848e+07,23198.456236,3.544758e+07,17467.176575


In [59]:
stats_borr = adv.statistical_validation(df_3_borrow, borrow_agg_cols, save=False)

# features -> rows, metrics -> columns
metric_cols = ["mean", "median", "std", "cv", "p95", "p99"]
stats = stats_borr.set_index("column")[metric_cols]

print(stats)

                                  mean        median           std        cv  \
column                                                                         
borrow_tx_count           1.056635e+02  9.500000e+01  5.206797e+01  0.492772   
repay_tx_count            8.886653e+01  7.200000e+01  7.180547e+01  0.808015   
variable_borrow_tx_count  1.056635e+02  9.500000e+01  5.206797e+01  0.492772   
unique_borrowers          7.298832e+01  6.700000e+01  3.042017e+01  0.416781   
unique_repayers           5.416071e+01  4.500000e+01  4.478458e+01  0.826883   
borrow_amount_value_usd   4.651291e+07  3.031552e+07  5.843244e+07  1.256263   
borrow_amount_value_eth   1.578487e+04  1.055787e+04  1.866181e+04  1.182259   
repay_amount_value_usd    4.565728e+07  2.850048e+07  5.848805e+07  1.281024   
repay_amount_value_eth    1.554396e+04  1.000056e+04  1.894298e+04  1.218671   

                                   p95           p99  
column                                                
borrow_tx

In [60]:
TABLE = "reserve_state_rates"  

df_0_reserve_state_rates = tf.load_table(TABLE)
df_0_reserve_state_rates.rename(columns={"update_count": "update_count_liquidity"}, inplace=True)
print(f"{TABLE}: {df_0_reserve_state_rates.shape[0]} rows x {df_0_reserve_state_rates.shape[1]} cols")
display(df_0_reserve_state_rates.dtypes.to_frame("dtype"))
display(df_0_reserve_state_rates.head(PREVIEW_ROWS))

reserve_state_rates: 99143 rows x 8 cols


,dtype
time_bucket,str
asset,str
liquidity_rate,float64
variable_borrow_rate,float64
stable_borrow_rate,int64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64


,time_bucket,asset,liquidity_rate,variable_borrow_rate,stable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,0.000240,0.006838,0,1.001884,1.016139,5
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,0.000260,0.005101,0,1.003301,1.022170,21
2,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,0.000000,0.045000,0,1.000000,1.123769,5
3,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,0.011037,0.033222,0,1.038794,1.074808,5
4,2025-04-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,0.000246,0.006911,0,1.000689,1.007202,2
5,2025-04-01 00:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,0.070883,0.108151,0,1.103381,1.165427,2
6,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,0.026601,0.048095,0,1.121824,1.179228,2
7,2025-04-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,0.011034,0.044527,0,1.053432,1.095327,6
8,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,0.001172,0.004968,0,1.001093,1.009286,6
9,2025-04-01 00:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,0.000000,0.000000,0,1.000000,1.000000,4


In [61]:
reserve_cols = [
    "liquidity_rate",
    "stable_borrow_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

# for col in reserve_cols:
# ALREADY DONE BY SQL
#     df_0_reserve_state_rates[col] = df_0_reserve_state_rates[col].astype(float) / 10**27

DF_common_1 = DF_common.merge(
    df_0_reserve_state_rates,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_1.drop(["stable_borrow_rate"], axis=1, inplace=True)
DF_common_1

,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045000,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
...,...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.034642,0.024840,0.034642,1.165281,1.222860,114
54194,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.022331,0.016594,0.022331,1.061964,1.094464,72
54195,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,0.022250,0.007921,0.022250,1.013739,1.031589,2
54196,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,0.003051,0.000017,0.003051,1.000195,1.004011,6


In [62]:
common_cols = [
    "last_borrow_rate",
    "liquidity_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

stats = pd.DataFrame({
    "mean": DF_common_1[common_cols].mean(),
    "median": DF_common_1[common_cols].median(),
    "std": DF_common_1[common_cols].std(),
    "variance": DF_common_1[common_cols].var(),
    "cv": DF_common_1[common_cols].std() / DF_common_1[common_cols].mean(),
    "p95": DF_common_1[common_cols].quantile(0.95),
    "p99": DF_common_1[common_cols].quantile(0.99)
})

print(stats)

                           mean    median       std  variance        cv  \
last_borrow_rate       0.035050  0.036207  0.038025  0.001446  1.084895   
liquidity_rate         0.015778  0.012786  0.019023  0.000362  1.205640   
variable_borrow_rate   0.033300  0.035450  0.027096  0.000734  0.813693   
liquidity_index        1.049320  1.019381  0.057466  0.003302  0.054765   
variable_borrow_index  1.094467  1.080621  0.080826  0.006533  0.073850   

                            p95       p99  
last_borrow_rate       0.062958  0.106419  
liquidity_rate         0.042809  0.054291  
variable_borrow_rate   0.063259  0.097939  
liquidity_index        1.154338  1.164819  
variable_borrow_index  1.222507  1.252292  


In [63]:
TABLE = "query_7804264" 

df_0_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_reserve_config.shape[0]} rows x {df_0_reserve_config.shape[1]} cols")
display(df_0_reserve_config.dtypes.to_frame("dtype"))
display(df_0_reserve_config.head(PREVIEW_ROWS))
summary = pd.DataFrame({
    "null_count": df_0_reserve_config.isna().sum(),
    "null_pct": df_0_reserve_config.isna().mean() * 100,
})
print(summary)

# reserve config results is broken, not enough nor consistent data with very high percentage of null values, so we will not use it for now.
# still compromised. the null values has been replaced with 0 and 1, but the data is not reliable enough to be used confidently in the analysis.

query_7804264: 232879 rows x 11 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
supply_cap,int64
borrow_cap,int64
debt_ceiling,float64
reserve_factor,float64
liquidation_threshold,float64
liquidation_bonus,float64
ltv,float64


,time_bucket,asset,asset_symbol,supply_cap,borrow_cap,debt_ceiling,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,config_event_count
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,9000000,475200,4500000.0,0.20,0.67,1.075,0.57,0
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,2200,275,0.0,0.50,0.78,1.075,0.73,0
2,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,6000000,330000,17000000.0,0.20,0.74,1.100,0.65,0
3,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,47500,28000,0.0,0.50,0.78,1.050,0.73,0
4,2025-04-01 00:00:00,0x3432b6a60d23ca0dfca7761b7ab56459d9c964d0,FXS,1200000,330000,4000000.0,0.20,0.42,1.100,0.00,0
5,2025-04-01 00:00:00,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0,192500000,0.0,0.00,0.00,0.000,0.00,0
6,2025-04-01 00:00:00,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,120000000,110000000,50000000.0,0.25,0.75,1.085,0.72,0
7,2025-04-01 00:00:00,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,20000000,13000000,0.0,0.20,0.71,1.070,0.66,0
8,2025-04-01 00:00:00,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,8000000,1500000,7500000.0,0.20,0.50,1.090,0.40,0
9,2025-04-01 00:00:00,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,7000000,5000000,0.0,0.20,0.77,1.045,0.00,0


                       null_count  null_pct
time_bucket                     0       0.0
asset                           0       0.0
asset_symbol                    0       0.0
supply_cap                      0       0.0
borrow_cap                      0       0.0
debt_ceiling                    0       0.0
reserve_factor                  0       0.0
liquidation_threshold           0       0.0
liquidation_bonus               0       0.0
ltv                             0       0.0
config_event_count              0       0.0


In [64]:
df_oracle_price_usd_eth

,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
205487,2026-03-30 22:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
205488,2026-03-30 22:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
205489,2026-03-30 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000145,0.000493,2
205490,2026-03-30 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [65]:
TABLE = "query_7711290"

df_1_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_1_reserve_config.shape[0]} rows x {df_1_reserve_config.shape[1]} cols")
display(df_1_reserve_config.dtypes.to_frame("dtype"))
# display(df_1_reserve_config.head(PREVIEW_ROWS))
df_1_reserve_config

query_7711290: 82 rows x 5 cols


,dtype
asset,str
asset_symbol,str
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,borrow_cap,0,whole_tokens
1,NaN,NaN,config_event_count,0,count
2,NaN,NaN,debt_ceiling,2,usd_2dp
3,NaN,NaN,latest_liquidation_block,0,block_number
4,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,liquidated_collateral_raw,18,raw_token_amount
...,...,...,...,...,...
77,NaN,NaN,receive_atoken_count,0,count
78,NaN,NaN,reserve_factor,4,basis_points
79,NaN,NaN,supply_cap,0,whole_tokens
80,NaN,NaN,unique_liquidated_users,0,count


In [66]:
# reserve_config is pre-normalized in normalize.ipynb — scaling removed here
df_2_reserve_config = df_0_reserve_config.copy()
df_2_reserve_config["debt_ceiling"] = df_2_reserve_config["debt_ceiling"] / 1e2
df_2_reserve_config

,time_bucket,asset,asset_symbol,supply_cap,borrow_cap,debt_ceiling,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,config_event_count
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,9000000,475200,45000.0,0.20,0.670,1.075,0.5700,0
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,2200,275,0.0,0.50,0.780,1.075,0.7300,0
2,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,6000000,330000,170000.0,0.20,0.740,1.100,0.6500,0
3,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,47500,28000,0.0,0.50,0.780,1.050,0.7300,0
4,2025-04-01 00:00:00,0x3432b6a60d23ca0dfca7761b7ab56459d9c964d0,FXS,1200000,330000,40000.0,0.20,0.420,1.100,0.0000,0
...,...,...,...,...,...,...,...,...,...,...,...
232874,2026-03-31 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,60000000,50000000,0.0,0.20,0.000,0.000,0.0000,0
232875,2026-03-31 22:00:00,0xe6a934089bbee34f832060ce98848359883749b3,PT-sUSDE-27NOV2025,1,1,0.0,0.45,0.001,1.075,0.0000,0
232876,2026-03-31 22:00:00,0xe8483517077afa11a9b07f849cee2552f040d7b2,PT-sUSDE-5FEB2026,720000000,1,0.0,0.45,0.001,1.075,0.0005,0
232877,2026-03-31 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,200000,1,0.0,0.15,0.750,1.075,0.0000,0


In [67]:
config_amount_cols = [
    "supply_cap",
    "borrow_cap",
    "debt_ceiling",
]

df_3_reserve_config = tf.multiply_by_price(df_2_reserve_config, df_oracle_price_usd_eth, config_amount_cols)
df_3_reserve_config




,time_bucket,asset,asset_symbol,supply_cap,borrow_cap,debt_ceiling,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,config_event_count,supply_cap_value_usd,supply_cap_value_eth,borrow_cap_value_usd,borrow_cap_value_eth,debt_ceiling_value_usd,debt_ceiling_value_eth
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,9000000,475200,45000.0,0.20,0.670,1.075,0.5700,0,1.691920e+06,9.260767e+02,8.933340e+04,4.889685e+01,8.459602e+03,4.630384
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,2200,275,0.0,0.50,0.780,1.075,0.7300,0,1.813408e+08,9.925730e+04,2.266760e+07,1.240716e+04,0.000000e+00,0.000000
2,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,6000000,330000,170000.0,0.20,0.740,1.100,0.6500,0,3.597500e+07,1.969100e+04,1.978625e+06,1.083005e+03,1.019292e+06,557.911790
3,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,47500,28000,0.0,0.50,0.780,1.050,0.7300,0,3.921321e+09,2.146344e+06,2.311516e+09,1.265213e+06,0.000000e+00,0.000000
4,2025-04-01 00:00:00,0x3432b6a60d23ca0dfca7761b7ab56459d9c964d0,FXS,1200000,330000,40000.0,0.20,0.420,1.100,0.0000,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232874,2026-03-31 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,60000000,50000000,0.0,0.20,0.000,0.000,0.0000,0,NaN,NaN,NaN,NaN,NaN,NaN
232875,2026-03-31 22:00:00,0xe6a934089bbee34f832060ce98848359883749b3,PT-sUSDE-27NOV2025,1,1,0.0,0.45,0.001,1.075,0.0000,0,NaN,NaN,NaN,NaN,NaN,NaN
232876,2026-03-31 22:00:00,0xe8483517077afa11a9b07f849cee2552f040d7b2,PT-sUSDE-5FEB2026,720000000,1,0.0,0.45,0.001,1.075,0.0005,0,NaN,NaN,NaN,NaN,NaN,NaN
232877,2026-03-31 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,200000,1,0.0,0.15,0.750,1.075,0.0000,0,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
config_amount_cols = [
    "supply_cap_value_usd",
    "supply_cap_value_eth",
    "borrow_cap_value_usd",
    "borrow_cap_value_eth",
    "debt_ceiling_value_usd",
    "debt_ceiling_value_eth",
]

df_4_reserve_config = (
    df_3_reserve_config
    [["time_bucket"] + config_amount_cols]
)

df_4_reserve_config_final = tf.aggregate_by_time_bucket(
    df_4_reserve_config,
    "time_bucket",
    config_amount_cols,
    agg_func="sum"
)

df_4_reserve_config_final

,time_bucket,supply_cap_value_usd,supply_cap_value_eth,borrow_cap_value_usd,borrow_cap_value_eth,debt_ceiling_value_usd,debt_ceiling_value_eth
0,2025-04-01 00:00:00,3.284362e+10,1.797703e+07,1.857387e+10,1.016645e+07,1.581227e+08,86548.907562
1,2025-04-01 02:00:00,3.293718e+10,1.794440e+07,1.861319e+10,1.014060e+07,1.580937e+08,86130.444958
2,2025-04-01 04:00:00,3.298993e+10,1.791995e+07,1.863489e+10,1.012237e+07,1.580543e+08,85853.775096
3,2025-04-01 06:00:00,3.310944e+10,1.785814e+07,1.868407e+10,1.007758e+07,1.597151e+08,86144.806127
4,2025-04-01 08:00:00,3.336168e+10,1.775156e+07,1.879091e+10,9.998550e+06,1.609780e+08,85655.535041
...,...,...,...,...,...,...,...
4375,2026-03-31 14:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
4376,2026-03-31 16:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
4377,2026-03-31 18:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
4378,2026-03-31 20:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000


In [69]:
df_3_reserve_config.drop(["asset_symbol", "supply_cap", "borrow_cap", "debt_ceiling", "config_event_count"], axis=1, inplace=True)
df_3_reserve_config = dv.canonicalize_keys(df_3_reserve_config)   # canonical time_bucket so the merge matches the panel
DF_common_2 = DF_common_1.merge(
    df_3_reserve_config,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_2

,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,supply_cap_value_usd,supply_cap_value_eth,borrow_cap_value_usd,borrow_cap_value_eth,debt_ceiling_value_usd,debt_ceiling_value_eth
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21,0.50,0.78,1.050,0.730,3.921321e+09,2.146344e+06,2.311516e+09,1.265213e+06,0.000000,0.000000
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045000,0.000000,0.045000,1.000000,1.123769,5,0.00,0.00,0.000,0.000,0.000000e+00,0.000000e+00,1.925831e+08,1.054108e+05,0.000000,0.000000
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5,0.25,0.75,1.085,0.720,1.200736e+08,6.572257e+04,1.100675e+08,6.024569e+04,500306.708333,273.844023
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2,0.25,0.77,1.050,0.630,3.382174e+08,1.851241e+05,2.711743e+08,1.484279e+05,0.000000,0.000000
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6,0.05,0.81,1.060,0.785,3.501046e+09,1.916305e+06,1.050314e+09,5.748916e+05,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.034642,0.024840,0.034642,1.165281,1.222860,114,0.10,0.78,1.045,0.750,7.500422e+09,3.696657e+06,7.000394e+09,3.450214e+06,0.000000,0.000000
54194,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.022331,0.016594,0.022331,1.061964,1.094464,72,0.15,0.83,1.050,0.805,7.710172e+09,3.800000e+06,7.304373e+09,3.600000e+06,0.000000,0.000000
54195,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,0.022250,0.007921,0.022250,1.013739,1.031589,2,0.20,0.00,0.000,0.000,2.998549e+08,1.477864e+05,1.399323e+08,6.896700e+04,0.000000,0.000000
54196,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,0.003051,0.000017,0.003051,1.000195,1.004011,6,0.50,0.78,1.075,0.730,2.134335e+09,1.051925e+06,9.604509e+07,4.733662e+04,0.000000,0.000000


In [74]:
TABLE = "liquidation"

df_0_liq = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_liq.shape[0]} rows x {df_0_liq.shape[1]} cols")
display(df_0_liq.dtypes.to_frame("dtype"))
# display(df_0_liq.head(PREVIEW_ROWS))
df_0_liq

liquidation: 4474 rows x 12 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
liquidated_collateral_raw,object
liquidation_debt_covered_raw,object
as_collateral_tx_count,int64
as_debt_tx_count,int64
liquidation_tx_count,int64
receive_atoken_count,int64
unique_liquidated_users,int64


,time_bucket,asset,asset_symbol,liquidated_collateral_raw,liquidation_debt_covered_raw,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,receive_atoken_count,unique_liquidated_users,unique_liquidators,latest_liquidation_block
0,2025-04-01 06:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,17432984242,0,1,0,1,0,1,1,22172329
1,2025-04-01 06:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0,10941402645440751977,0,2,2,0,2,2,22172329
2,2025-04-01 06:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,4541455,0,1,0,1,0,1,1,22172327
3,2025-04-01 08:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,0,2836667175746209741782,0,1,1,0,1,1,22172888
4,2025-04-01 08:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,41214339648,0,1,0,1,0,1,1,22172888
...,...,...,...,...,...,...,...,...,...,...,...,...
4469,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,37879250,0,1,0,1,0,1,1,24774338
4470,2026-03-31 06:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0,1071422,0,11,11,0,11,1,24776085
4471,2026-03-31 06:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,545495972674491,0,11,0,11,0,11,1,24776085
4472,2026-03-31 20:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,366,0,1,0,1,0,1,1,24780310


In [71]:
TABLE = "query_7711290" 

df_1_liq = tf.load_table(TABLE)
print(f"{TABLE}: {df_1_liq.shape[0]} rows x {df_1_liq.shape[1]} cols")
display(df_1_liq.dtypes.to_frame("dtype"))
display(df_1_liq.head(PREVIEW_ROWS))

query_7711290: 82 rows x 5 cols


,dtype
asset,str
asset_symbol,str
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,borrow_cap,0,whole_tokens
1,NaN,NaN,config_event_count,0,count
2,NaN,NaN,debt_ceiling,2,usd_2dp
3,NaN,NaN,latest_liquidation_block,0,block_number
4,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,liquidated_collateral_raw,18,raw_token_amount
5,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,liquidated_collateral_raw,18,raw_token_amount
6,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,liquidated_collateral_raw,18,raw_token_amount
7,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,liquidated_collateral_raw,8,raw_token_amount
8,0x3b3fb9c57858ef816833dc91565efcd85d96f634,PT-sUSDE-31JUL2025,liquidated_collateral_raw,18,raw_token_amount
9,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,liquidated_collateral_raw,18,raw_token_amount


In [72]:
df_2_liq = tf.scale_by_decimals(df_0_liq, df_oracle_price_usd_eth, drop_raw=True)
df_3_liq = tf.multiply_by_price(df_2_liq, df_oracle_price_usd_eth, ["liquidated_collateral", "liquidation_debt_covered"])
df_3_liq


,time_bucket,asset,asset_symbol,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,receive_atoken_count,unique_liquidated_users,unique_liquidators,latest_liquidation_block,liquidated_collateral,liquidation_debt_covered,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth
0,2025-04-01 06:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1,0,1,0,1,1,22172329,17432.984242,0.000000,17444.527057,9.409033,0.000000,0.000000
1,2025-04-01 06:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0,2,2,0,2,2,22172329,0.000000,10.941403,0.000000,0.000000,20285.665952,10.941403
2,2025-04-01 06:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,1,0,1,0,1,1,22172327,0.045415,0.000000,3785.518840,2.041781,0.000000,0.000000
3,2025-04-01 08:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,0,1,1,0,1,1,22172888,0.000000,2836.667176,0.000000,0.000000,39643.605726,21.094034
4,2025-04-01 08:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1,0,1,0,1,1,22172888,41214.339648,0.000000,41239.021886,21.943161,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4469,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1,0,1,0,1,1,24774338,37.879250,0.000000,NaN,NaN,NaN,NaN
4470,2026-03-31 06:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0,11,11,0,11,1,24776085,0.000000,1.071422,NaN,NaN,NaN,NaN
4471,2026-03-31 06:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,11,0,11,0,11,1,24776085,0.000545,0.000000,NaN,NaN,NaN,NaN
4472,2026-03-31 20:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,1,0,1,0,1,1,24780310,0.000004,0.000000,NaN,NaN,NaN,NaN


In [73]:
df_3_liq.drop(["latest_liquidation_block"], axis=1, inplace=True)
liq_agg_cols = [
    "as_collateral_tx_count",
    "as_debt_tx_count",
    "liquidation_tx_count",
    "unique_liquidated_users",
    "unique_liquidators",
    "liquidated_collateral_value_usd",
    "liquidated_collateral_value_eth",
    "liquidation_debt_covered_value_usd",
    "liquidation_debt_covered_value_eth",
]
df_4_liq = tf.aggregate_by_time_bucket(df_3_liq, "time_bucket", liq_agg_cols, agg_func='sum')
df_4_liq

,time_bucket,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,unique_liquidated_users,unique_liquidators,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth
0,2025-04-01 06:00:00.000 UTC,2,2,4,4,4,21230.045897,11.450814,20285.665952,10.941403
1,2025-04-01 08:00:00.000 UTC,1,1,2,2,2,41239.021886,21.943161,39643.605726,21.094034
2,2025-04-01 14:00:00.000 UTC,2,2,4,4,4,20970.444109,11.118519,19269.584636,10.218210
3,2025-04-01 20:00:00.000 UTC,1,1,2,2,2,17.627285,0.009220,16.855512,0.008816
4,2025-04-02 00:00:00.000 UTC,1,1,2,2,2,2716.800395,1.433326,2524.128530,1.331694
...,...,...,...,...,...,...,...,...,...,...
1220,2026-03-30 18:00:00.000 UTC,1,1,2,2,2,1178.444683,0.580937,1130.882604,0.557495
1221,2026-03-30 22:00:00.000 UTC,2,2,4,4,4,26663.263046,13.141165,25279.567462,12.459286
1222,2026-03-31 00:00:00.000 UTC,1,1,2,2,2,0.000000,0.000000,0.000000,0.000000
1223,2026-03-31 06:00:00.000 UTC,11,11,22,22,2,0.000000,0.000000,0.000000,0.000000


In [75]:
TABLE = "collateral_toggle"

df_0_collateral = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_collateral.shape[0]} rows x {df_0_collateral.shape[1]} cols")
display(df_0_collateral.dtypes.to_frame("dtype"))
display(df_0_collateral.head(PREVIEW_ROWS))

collateral_toggle: 60580 rows x 8 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
collateral_enabled_count,int64
collateral_disabled_count,int64
unique_collateral_enable_users,int64
unique_collateral_disable_users,int64
latest_collateral_toggle_block,int64


,time_bucket,asset,asset_symbol,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users,latest_collateral_toggle_block
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,1,1,1,1,22170819
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,9,4,8,3,22170740
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,1,0,1,0,22170829
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0,1,0,1,22170539
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2,2,2,2,22170805
5,2025-04-01 00:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,3,4,3,4,22170696
6,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,18,18,12,13,22170880
7,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,32,24,20,12,22170909
8,2025-04-01 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,0,2,0,2,22170828
9,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,24,31,11,17,22170925


In [76]:
collateral_agg_cols = [
    "collateral_enabled_count",
    "collateral_disabled_count",
    "unique_collateral_enable_users",
    "unique_collateral_disable_users"
]

df_1_collateral = tf.aggregate_by_time_bucket(df_0_collateral, "time_bucket", collateral_agg_cols, agg_func='sum')
df_1_collateral

,time_bucket,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users
0,2025-04-01 00:00:00.000 UTC,91,88,59,56
1,2025-04-01 02:00:00.000 UTC,86,64,61,41
2,2025-04-01 04:00:00.000 UTC,65,59,41,36
3,2025-04-01 06:00:00.000 UTC,92,72,58,39
4,2025-04-01 08:00:00.000 UTC,110,83,68,43
...,...,...,...,...,...
4375,2026-03-31 14:00:00.000 UTC,55,48,46,38
4376,2026-03-31 16:00:00.000 UTC,47,59,31,43
4377,2026-03-31 18:00:00.000 UTC,64,43,54,35
4378,2026-03-31 20:00:00.000 UTC,48,54,40,44


In [77]:
TABLE = "flashloan"

df_0_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_flashloan.shape[0]} rows x {df_0_flashloan.shape[1]} cols")
display(df_0_flashloan.dtypes.to_frame("dtype"))
display(df_0_flashloan.head(PREVIEW_ROWS))

flashloan: 24940 rows x 11 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
flashloan_amount,float64
flashloan_premium,float64
flashloan_tx_count,int64
unique_flashloan_initiators,int64
no_open_debt_flashloan_tx_count,int64
stable_flashloan_tx_count,int64
variable_flashloan_tx_count,int64


,time_bucket,asset,asset_symbol,flashloan_amount,flashloan_premium,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,stable_flashloan_tx_count,variable_flashloan_tx_count,latest_flashloan_block
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,2.023045,0.001012,1,1,1,0,0,22170544
1,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,26.964624,0.013482,1,1,1,0,0,22170805
2,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,239703.808517,87.236780,12,9,10,0,2,22170873
3,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,17899.829513,0.000000,1,1,0,0,1,22170789
4,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,189.907915,0.012555,6,3,3,0,3,22170930
5,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,128282.362697,24.976946,5,5,4,0,1,22170443
6,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,72.449434,0.036225,4,4,4,0,0,22171433
7,2025-04-01 02:00:00.000 UTC,0x83f20f44975d03b1b09e64809b757c47f942beea,sDAI,17652.792226,0.000000,1,1,1,0,0,22171301
8,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,270225.012614,4.468111,7,5,5,0,2,22171321
9,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,100.768951,0.008320,4,4,3,0,1,22171400


In [78]:
TABLE = "query_7711298"

df_2_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_2_flashloan.shape[0]} rows x {df_2_flashloan.shape[1]} cols")
# display(df_2_flashloan.dtypes.to_frame("dtype"))
# display(df_2_flashloan.head(PREVIEW_ROWS))

# split the reference frame by its `metric` column
df_2_flashloan_amount  = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_amount_raw"].copy()
df_2_flashloan_premium = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_premium_raw"].copy()

display(df_2_flashloan_amount.head(PREVIEW_ROWS))
display(df_2_flashloan_premium.head(PREVIEW_ROWS))

query_7711298: 95 rows x 5 cols


,asset,asset_symbol,metric,decimals,unit
0,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_amount_raw,18,raw_token_amount
1,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_amount_raw,18,raw_token_amount
2,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_amount_raw,6,raw_token_amount
3,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_amount_raw,18,raw_token_amount
4,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_amount_raw,18,raw_token_amount
5,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_amount_raw,8,raw_token_amount
6,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_amount_raw,6,raw_token_amount
7,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_amount_raw,18,raw_token_amount
8,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_amount_raw,18,raw_token_amount
9,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_amount_raw,18,raw_token_amount


,asset,asset_symbol,metric,decimals,unit
43,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_premium_raw,18,raw_token_amount
44,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_premium_raw,18,raw_token_amount
45,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_premium_raw,6,raw_token_amount
46,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_premium_raw,18,raw_token_amount
47,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_premium_raw,18,raw_token_amount
48,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_premium_raw,8,raw_token_amount
49,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_premium_raw,6,raw_token_amount
50,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_premium_raw,18,raw_token_amount
51,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_premium_raw,18,raw_token_amount
52,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_premium_raw,18,raw_token_amount


In [79]:
flashloan_agg_cols = [
    "flashloan_tx_count",
    "unique_flashloan_initiators",
    "no_open_debt_flashloan_tx_count",
    "variable_flashloan_tx_count",
]

df_1_flashloan = tf.aggregate_by_time_bucket(df_0_flashloan, "time_bucket", flashloan_agg_cols, agg_func='sum')
df_1_flashloan

,time_bucket,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 00:00:00.000 UTC,26,20,19,7
1,2025-04-01 02:00:00.000 UTC,23,19,19,4
2,2025-04-01 04:00:00.000 UTC,9,7,8,1
3,2025-04-01 06:00:00.000 UTC,17,7,6,11
4,2025-04-01 08:00:00.000 UTC,28,17,17,11
...,...,...,...,...,...
4368,2026-03-31 14:00:00.000 UTC,22,9,22,0
4369,2026-03-31 16:00:00.000 UTC,15,7,15,0
4370,2026-03-31 18:00:00.000 UTC,13,6,12,1
4371,2026-03-31 20:00:00.000 UTC,17,7,17,0


In [80]:
df_2_flashloan_scaled = tf.scale_by_decimals(df_0_flashloan, df_2_flashloan_amount, drop_raw=True)
df_2_flashloan_scaled.drop(flashloan_agg_cols, axis=1, inplace=True)
df_2_flashloan_scaled.drop(["stable_flashloan_tx_count", "latest_flashloan_block"], axis=1, inplace=True)
df_2_flashloan_scaled_1 = tf.multiply_by_price(df_2_flashloan_scaled, df_oracle_price_usd_eth, ["flashloan_amount", "flashloan_premium"])
# df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')

stats_flashloan = [
    "flashloan_amount_value_usd",
    "flashloan_amount_value_eth",
    "flashloan_premium_value_usd",
    "flashloan_premium_value_eth",
]

df_2_flashloan_scaled_1



,time_bucket,asset,asset_symbol,flashloan_amount,flashloan_premium,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,2.023045,0.001012,167010.705822,91.413693,83.505152,0.045707
1,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,26.964624,0.013482,59002.743704,32.295284,29.501372,0.016148
2,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,239703.808517,87.236780,239867.236576,131.291881,87.296257,0.047782
3,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,17899.829513,0.000000,13552.237626,7.417868,0.000000,0.000000
4,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,189.907915,0.012555,346957.408685,189.907915,22.937446,0.012555
...,...,...,...,...,...,...,...,...,...
24935,2026-03-31 20:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,10016.402767,5.008202,NaN,NaN,NaN,NaN
24936,2026-03-31 20:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,0.238093,0.000119,NaN,NaN,NaN,NaN
24937,2026-03-31 22:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,16914.441044,8.457221,NaN,NaN,NaN,NaN
24938,2026-03-31 22:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,28.311710,0.014156,NaN,NaN,NaN,NaN


In [81]:
df_2_flashloan_scaled_2 = tf.aggregate_by_time_bucket(df_2_flashloan_scaled_1, "time_bucket", stats_flashloan , agg_func='sum')
df_2_flashloan_scaled_2


,time_bucket,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth
0,2025-04-01 00:00:00.000 UTC,9.547419e+05,522.580146,248.230641,0.135870
1,2025-04-01 02:00:00.000 UTC,6.532868e+06,3559.148456,3038.132814,1.655194
2,2025-04-01 04:00:00.000 UTC,2.605000e+05,141.502472,8.453992,0.004592
3,2025-04-01 06:00:00.000 UTC,1.136226e+06,612.843849,147.875707,0.079760
4,2025-04-01 08:00:00.000 UTC,2.134066e+06,1135.526324,708.085451,0.376769
...,...,...,...,...,...
4368,2026-03-31 14:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000
4369,2026-03-31 16:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000
4370,2026-03-31 18:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000
4371,2026-03-31 20:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000


In [82]:
DF_flashloan_common = df_2_flashloan_scaled_2.merge(
    df_1_flashloan,
      on=["time_bucket"], how="left", suffixes=("_x", "_y"))

DF_flashloan_common

,time_bucket,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 00:00:00.000 UTC,9.547419e+05,522.580146,248.230641,0.135870,26,20,19,7
1,2025-04-01 02:00:00.000 UTC,6.532868e+06,3559.148456,3038.132814,1.655194,23,19,19,4
2,2025-04-01 04:00:00.000 UTC,2.605000e+05,141.502472,8.453992,0.004592,9,7,8,1
3,2025-04-01 06:00:00.000 UTC,1.136226e+06,612.843849,147.875707,0.079760,17,7,6,11
4,2025-04-01 08:00:00.000 UTC,2.134066e+06,1135.526324,708.085451,0.376769,28,17,17,11
...,...,...,...,...,...,...,...,...,...
4368,2026-03-31 14:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000,22,9,22,0
4369,2026-03-31 16:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000,15,7,15,0
4370,2026-03-31 18:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000,13,6,12,1
4371,2026-03-31 20:00:00.000 UTC,0.000000e+00,0.000000,0.000000,0.000000,17,7,17,0


In [83]:
TABLE = "user_account"

df_0_user_account = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account.shape[0]} rows x {df_0_user_account.shape[1]} cols")
display(df_0_user_account.dtypes.to_frame("dtype"))
display(df_0_user_account.head(PREVIEW_ROWS))
df_0_user_account.drop(["min_health_factor", "max_health_factor"], axis=1, inplace=True)

user_account: 2181 rows x 10 cols


,dtype
time_bucket,str
avg_total_collateral_base,float64
avg_total_debt_base,float64
avg_available_borrows_base,float64
avg_current_liquidation_threshold,float64
avg_ltv,float64
min_health_factor,float64
max_health_factor,float64
sampled_user_count,int64
account_data_call_count,int64


,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,min_health_factor,max_health_factor,sampled_user_count,account_data_call_count
0,2025-04-01 02:00:00.000 UTC,1.491521e+16,4.997479e+15,6.710961e+15,8100.0,7850.000000,2.417483e+18,2.417483e+18,1,1
1,2025-04-01 04:00:00.000 UTC,1.506325e+16,5.192650e+15,6.631999e+15,8100.0,7850.000000,2.349712e+18,2.349712e+18,1,1
2,2025-04-01 08:00:00.000 UTC,1.262487e+11,4.772150e+10,4.696503e+10,7800.0,7500.000000,2.063514e+18,2.063514e+18,1,1
3,2025-04-01 12:00:00.000 UTC,1.047713e+16,6.422148e+15,3.321578e+15,9500.0,9300.000000,1.549835e+18,1.549835e+18,1,1
4,2025-04-02 12:00:00.000 UTC,1.047771e+16,6.422465e+15,3.321807e+15,9500.0,9300.000000,1.549845e+18,1.549845e+18,1,1
5,2025-04-02 22:00:00.000 UTC,8.024767e+11,5.746264e+11,2.643676e+10,7793.0,7271.666667,1.004146e+18,1.148862e+18,3,3
6,2025-04-03 10:00:00.000 UTC,1.478903e+16,4.509097e+15,7.100292e+15,8100.0,7850.000000,2.656655e+18,2.656655e+18,1,1
7,2025-04-03 12:00:00.000 UTC,5.965423e+14,3.615911e+14,1.875464e+14,7799.5,7534.055556,1.042772e+18,1.157921e+77,9,18
8,2025-04-03 14:00:00.000 UTC,1.970535e+13,1.336692e+13,1.571601e+12,7985.5,7581.500000,1.153408e+18,1.204179e+18,1,2
9,2025-04-03 20:00:00.000 UTC,6.572538e+13,5.917612e+13,1.948483e+12,9500.0,9300.000000,1.054971e+18,1.055556e+18,1,2


In [84]:
stats_user = [
    df_0_user_account["sampled_user_count"].mean(), 
    df_0_user_account["sampled_user_count"].median(),
    # df_0_user_account["sampled_user_count"].std(),
    # df_0_user_account["sampled_user_count"].var(),
    df_0_user_account["account_data_call_count"].mean(),
    df_0_user_account["account_data_call_count"].median(),
#   df_0_user_account["account_data_call_count"].std(),
#   df_0_user_account["account_data_call_count"].var()
]

print(stats_user)

[np.float64(2.2576799633195783), np.float64(1.0), np.float64(6.20128381476387), np.float64(3.0)]


In [85]:
TABLE = "query_7711304"

df_0_user_account_agg = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account_agg.shape[0]} rows x {df_0_user_account_agg.shape[1]} cols")
display(df_0_user_account_agg.dtypes.to_frame("dtype"))
# display(df_0_user_account_agg.head(PREVIEW_ROWS))

df_0_user_account_agg_1 = df_0_user_account_agg.drop(df_0_user_account_agg.index[[0, 8]])
df_0_user_account_agg_1


query_7711304: 9 rows x 5 cols


,dtype
asset,float64
asset_symbol,float64
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
1,NaN,NaN,avg_available_borrows_base,8,usd_base_8dp
2,NaN,NaN,avg_current_liquidation_threshold,4,basis_points
3,NaN,NaN,avg_ltv,4,basis_points
4,NaN,NaN,avg_total_collateral_base,8,usd_base_8dp
5,NaN,NaN,avg_total_debt_base,8,usd_base_8dp
6,NaN,NaN,max_health_factor,18,wad
7,NaN,NaN,min_health_factor,18,wad


In [86]:
metric_cols = dict(zip(
    df_0_user_account_agg_1["metric"],
    df_0_user_account_agg_1["decimals"]
))

    
out = df_0_user_account.copy()

for col, dec in metric_cols.items():
    if col not in out.columns:
        continue  # skip metrics that aren't columns in this frame
    out[col] = out[col].astype("float64") / (10 ** int(dec))
df_0_user_account_agg_scaled = out
df_0_user_account_agg_scaled

,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count
0,2025-04-01 02:00:00.000 UTC,1.491521e+08,4.997479e+07,6.710961e+07,0.810,0.7850,1,1
1,2025-04-01 04:00:00.000 UTC,1.506325e+08,5.192650e+07,6.631999e+07,0.810,0.7850,1,1
2,2025-04-01 08:00:00.000 UTC,1.262487e+03,4.772150e+02,4.696503e+02,0.780,0.7500,1,1
3,2025-04-01 12:00:00.000 UTC,1.047713e+08,6.422148e+07,3.321578e+07,0.950,0.9300,1,1
4,2025-04-02 12:00:00.000 UTC,1.047771e+08,6.422465e+07,3.321807e+07,0.950,0.9300,1,1
...,...,...,...,...,...,...,...,...
2176,2026-03-31 12:00:00.000 UTC,7.885368e+00,2.084495e+00,5.248898e+00,0.950,0.9300,1,1
2177,2026-03-31 14:00:00.000 UTC,7.806912e+00,2.081363e+00,5.179064e+00,0.950,0.9300,1,4
2178,2026-03-31 16:00:00.000 UTC,2.437547e+04,1.726822e+04,2.354035e+03,0.415,0.4025,3,4
2179,2026-03-31 18:00:00.000 UTC,7.092377e+00,6.094455e+00,2.886851e-01,0.920,0.9000,1,1


In [87]:
# df_4_supply_withdraw
# df_3_borrow
# DF_common_1
# df_4_liq
# df_1_collateral
# DF_flashloan_common
# df_0_user_account_agg_scaled

DF_common_final = df_4_supply_withdraw.merge(
    df_3_borrow, on=["time_bucket"], how="left", suffixes=("", "_y")).merge(
    df_4_liq, on=["time_bucket"], how="left", suffixes=("", "_liq")).merge(
    df_1_collateral, on=["time_bucket"], how="left", suffixes=("", "_collateral")).merge(
    DF_flashloan_common, on=["time_bucket"], how="left", suffixes=("", "_flashloan")).merge(
    df_0_user_account_agg_scaled, on=["time_bucket"], how="left", suffixes=("", "_useragg"))

display(DF_common_final.head(PREVIEW_ROWS))
DF_common_final
display(DF_common_final.dtypes.to_frame("dtype"))
display(DF_common_1.head(PREVIEW_ROWS))


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count
0,2025-04-01 00:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287,54.0,...,20.0,19.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-01 02:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741,58.0,...,19.0,19.0,4.0,1.491521e+08,4.997479e+07,6.710961e+07,0.81,0.785,1.0,1.0
2,2025-04-01 04:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778,48.0,...,7.0,8.0,1.0,1.506325e+08,5.192650e+07,6.631999e+07,0.81,0.785,1.0,1.0
3,2025-04-01 06:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607,64.0,...,7.0,6.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-04-01 08:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327,95.0,...,17.0,17.0,11.0,1.262487e+03,4.772150e+02,4.696503e+02,0.78,0.750,1.0,1.0
5,2025-04-01 10:00:00.000 UTC,127,111,77,59,2.532615e+08,135086.519080,2.552678e+08,136156.675648,72.0,...,12.0,11.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-04-01 12:00:00.000 UTC,127,122,60,62,4.748889e+08,254080.767024,4.601955e+08,246219.340371,57.0,...,12.0,14.0,3.0,1.047713e+08,6.422148e+07,3.321578e+07,0.95,0.930,1.0,1.0
7,2025-04-01 14:00:00.000 UTC,142,111,66,60,5.869603e+08,311246.523260,4.781272e+08,253535.842057,98.0,...,21.0,17.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-04-01 16:00:00.000 UTC,122,103,62,55,1.321209e+08,68913.296488,1.362937e+08,71089.779751,82.0,...,13.0,11.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-04-01 18:00:00.000 UTC,137,98,54,55,5.251323e+08,275222.083697,5.202348e+08,272655.318074,97.0,...,16.0,25.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,dtype
time_bucket,str
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64
supply_amount_value_usd,float64
supply_amount_value_eth,float64
withdrawal_amount_value_usd,float64
withdrawal_amount_value_eth,float64
borrow_tx_count,float64


,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045000,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
5,2025-04-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.046031,0.029176,0.045980,1.125029,1.164874,91
6,2025-04-01 00:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,0.089417,0.017815,0.088239,1.023239,1.127670,3
7,2025-04-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.026735,0.020251,0.026735,1.042603,1.067234,96
8,2025-04-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,0.048260,0.032141,0.048260,1.119375,1.163539,73
9,2025-04-01 00:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,0.035663,0.054515,1.144648,1.189138,1


In [88]:
from pathlib import Path

OUT_DIR = Path("transformed_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DF_common_final.to_csv(OUT_DIR / "DF_common_final.csv", index=False)
DF_common_1.to_csv(OUT_DIR / "DF_common_1.csv", index=False)

OUT_DIR_1 = Path("processed_data")

processed_tables = [
    df_4_supply_withdraw,
    df_3_borrow,
    DF_common_1,
    df_4_liq,
    df_1_collateral,
    DF_flashloan_common,
    df_0_user_account_agg_scaled,
    ]

for i, df in enumerate(processed_tables):
    df.to_csv(OUT_DIR_1 / f"table_{i}.csv", index=False)
    print(f"wrote {len(df)} rows to {OUT_DIR_1}/")



print(f"wrote {len(DF_common_final)} + {len(DF_common_1)} rows to {OUT_DIR}/")

wrote 4380 rows to processed_data/
wrote 4368 rows to processed_data/
wrote 54198 rows to processed_data/
wrote 1225 rows to processed_data/
wrote 4380 rows to processed_data/
wrote 4373 rows to processed_data/
wrote 2181 rows to processed_data/
wrote 4380 + 54198 rows to transformed_data/
